# Receipt PDF Parsing with Multimodal

Extract structured data from multiple receipt PDFs into a single pandas DataFrame.

This example demonstrates:
- **Multimodal mode** for processing PDF files through the Files API
- **Pydantic structured output** for type-safe receipt parsing
- **Batch processing** across multiple files with a single `BatchResponses` call

## 1. Setup

In [ ]:
import tempfile
from pathlib import Path

import pandas as pd
from pydantic import BaseModel

import openaivec
from openaivec import BatchResponses

## 2. Define receipt schema

Using a Pydantic model ensures every receipt is parsed into the same structure.

In [ ]:
class ReceiptItem(BaseModel):
    """A single line item on a receipt."""
    description: str
    quantity: int
    unit_price: float
    amount: float


class Receipt(BaseModel):
    """Structured representation of a receipt."""
    store_name: str
    date: str
    items: list[ReceiptItem]
    subtotal: float
    tax: float
    total: float
    payment_method: str

## 3. Prepare receipt files

In a real scenario, you would point to actual PDF files on disk.  
Here we create text files as stand-ins for demonstration.  
With `multimodal=True`, text files are read and **batched together** in a single API call.

In [ ]:
tmpdir = tempfile.mkdtemp()

receipts_raw = {
    "receipt_coffee_shop.txt": """
SUNNY CAFE
123 Main Street, Tokyo
Date: 2025-03-15

Latte                x2    ¥550    ¥1,100
Croissant            x1    ¥380    ¥380
Blueberry Muffin     x3    ¥420    ¥1,260

Subtotal: ¥2,740
Tax (10%): ¥274
Total: ¥3,014
Payment: Credit Card (Visa)
""",
    "receipt_electronics.txt": """
TECH WORLD AKIHABARA
456 Electric Town, Tokyo
Date: 2025-03-16

USB-C Cable          x2    ¥980    ¥1,960
Wireless Mouse       x1    ¥3,500  ¥3,500
Screen Protector     x3    ¥650    ¥1,950

Subtotal: ¥7,410
Tax (10%): ¥741
Total: ¥8,151
Payment: Cash
""",
    "receipt_grocery.txt": """
FRESH MART SHIBUYA
789 Center Gai, Tokyo
Date: 2025-03-17

Organic Milk 1L      x2    ¥320    ¥640
Whole Wheat Bread     x1    ¥280    ¥280
Avocado              x4    ¥198    ¥792
Salmon Fillet 200g   x2    ¥580    ¥1,160

Subtotal: ¥2,872
Tax (8%): ¥230
Total: ¥3,102
Payment: IC Card (Suica)
""",
}

file_paths = []
for name, content in receipts_raw.items():
    path = Path(tmpdir) / name
    path.write_text(content)
    file_paths.append(str(path))

print(f"Created {len(file_paths)} receipt files")

## 4. Parse receipts with BatchResponses

With `multimodal=True`, text files (`.txt`, `.csv`, `.md`, …) are **read as strings**
and processed through the batched text path — giving you deduplication and
batch optimization for free.  
Binary files (`.pdf`, `.docx`, `.xlsx`) are automatically uploaded via the Files API.

In [ ]:
batch = BatchResponses.of(
    client=openaivec.get_client(),
    model_name=openaivec.get_responses_model(),
    system_message=(
        "You are a receipt parser. Extract all information from the receipt "
        "into the structured format. Use numeric values without currency symbols or commas. "
        "Dates should be in YYYY-MM-DD format."
    ),
    response_format=Receipt,
    batch_size=10,
    multimodal=True,
)

parsed: list[Receipt | None] = batch.parse(file_paths)

for i, receipt in enumerate(parsed):
    if receipt:
        print(f"Receipt {i+1}: {receipt.store_name} — ¥{receipt.total:,.0f}")

## 5. Build a line-item DataFrame

Flatten the structured results into one table — each row is a line item.

In [ ]:
rows = []
for receipt in parsed:
    if receipt is None:
        continue
    for item in receipt.items:
        rows.append({
            "store": receipt.store_name,
            "date": receipt.date,
            "item": item.description,
            "qty": item.quantity,
            "unit_price": item.unit_price,
            "amount": item.amount,
            "tax": receipt.tax,
            "total": receipt.total,
            "payment": receipt.payment_method,
        })

df = pd.DataFrame(rows)
df

## 6. Aggregate by store

In [ ]:
summary = (
    df.groupby("store")
    .agg(
        items=pd.NamedAgg(column="item", aggfunc="count"),
        total_amount=pd.NamedAgg(column="amount", aggfunc="sum"),
        total_with_tax=pd.NamedAgg(column="total", aggfunc="first"),
        payment=pd.NamedAgg(column="payment", aggfunc="first"),
    )
    .sort_values("total_with_tax", ascending=False)
)
summary

## 7. Cleanup

In [ ]:
import shutil
shutil.rmtree(tmpdir)
print("Done!")

## Notes

- **Text files** (`.txt`, `.csv`, `.md`, `.json`, etc.) are read as strings
  and processed through the **batched text path** — benefiting from
  deduplication and batch optimization.
- **Binary files** (`.pdf`, `.docx`, `.xlsx`, etc.) are uploaded via the
  **Files API** and processed individually.
- The same `multimodal=True` flag handles both cases transparently.
- For real PDF receipts, simply pass the `.pdf` file paths — the Files API
  upload is automatic.